# 🌿 Green Chemistry Analysis

Authors: Antoine Tran, Anna Griffa, Elsa Chevalier, Thomas Clément

![Homepage screenshot](interface.jpg)
*Figure 1. Homepage of the Green Chemistry Analysis dashboard showing therapeutic category selection.*

# Goal statement 

Over the recent years, toxicity and environmental issues have raised a lot of concerns about the planet's long-term future. Chemists developed the concept of green chemistry to optimize chemical processes in a safer and more environmentally responsible way. We, as chemistry students and future chemists, aspire to contribute to a more sustainable planet, and green chemistry is a topic that strongly resonates with us.

Antihistamines represent an interesting case study for green chemistry analysis due to their widespread use in pharmaceutical manufacturing and the potential environmental impact associated with their synthesis. Comparing alternative production routes for these compounds allows us to assess how sustainability principles can be applied to real-world medicinal chemistry processes.

The goal of this project is to compare different synthesis methods of antihistamines from a green chemistry perspective. Several commonly used antihistamines covering five different categories of symptoms were selected for this comparison. Antihistamines were selected because they represent a widely used pharmaceutical class with multiple synthetic routes, making them suitable candidates for comparative sustainability analysis.

We compare the synthesis pathways using the following metrics:

1. E-Factor, which compares the mass of waste generated to the mass of the final pharmaceutical product.
2. Atom Economy, which measures how efficiently the reactant mass is incorporated into the desired final product.
3. Solvent hazard assessment, based on GHS hazard classifications of the solvents involved in the synthesis.

# Project method 

To achieve the described goal, the following features were developed:

1. Reaction Dataset Module: a structured reaction dataset that represents each antihistamine synthesis pathway and serves as the foundation of the analysis.

2. Molecular Property Dataset: A database containing the relevant chemical properties of the molecules involved in the reactions was integrated to support the different metric calculations and hazard assessment. The PubChem API was used to retrieve molecular identifiers and chemical structure representations for the compounds involved in each reaction.

3. Metric Calculation Engine: a computational module was developed to automatically calculate the quantitative green chemistry indicators for each synthesis pathway. An overall green score is ultimately assigned to each synthesis using the calculated indicators, as well as additional process-related information such as the number of synthesis steps and the reaction temperature.

4. Green Chemistry Principle Evaluation Module: a hybrid rule-based assessment system was developed to evaluate compliance with the twelve principles. Some principles assessment depends on the indicators previously calculated whereas some are predefined and found in the dataset. This provides an additional qualitative assessment to the numerical metrics.

5. Graphical User Interface (GUI): An interactive Streamlit dashboard was developed to allow users to select symptom categories, compare antihistamine compounds, and visualize both quantitative sustainability metrics, qualitative green chemistry evaluations, and simplified synthesis equations. The interface was designed in an intuitive and user-friendly format.

# Core 

This section describes the main implementation components of the project, including the data modelling architecture, metric calculation engine, green chemistry evaluation logic, and graphical user interface.


```text
.
├── app.py                  # Main Streamlit application
├── data/
│   ├── reactions.json      # Reaction dataset
│   ├── molecules.json      # Molecular property dataset
│   ├── reactions.py        # Reaction data loading module
│   ├── molecules.py        # Molecular data loading module
│   └── principles.py
│
├── utils/
│   ├── metrics.py          # Sustainability metric calculations
│   ├── structures.py
│   ├── compare.py
│   └── scoring.py          # Green score computation
│
├── notebooks/              # Jupyter notebooks and project report
│  └── report
└── README.md

```


1. Reaction and Molecular Data Modelling Module: 

In order to store the detailed synthesis pathways, the file reactions.json compiles information including reactants, intermediates, products, solvents, reaction yield, operating temperature, synthesis steps, and therapeutic category. It serves as the core database of the project since these data are used throughout the project to perform the sustainability metric calculations, generate simplified reaction equations, rank synthesis pathways, and populate the interactive comparison interface.

To store characteristic properties of each compound involved in the previously described reactions, the file molecules.json was created and consists of the core of the quantitative analysis. It indeed supports all of the calculations by providing key properties such as molar mass, required for mass-based sustainability metrics, and GHS hazard classifications, which are used as a simplified proxy for solvent hazard assessment in the green chemistry evaluation.
The JSON structure provides a lightweight way to store structured chemical information easily accessible from Python. This information can then be loaded and processed through Python modules.


The PubChem API was used as a complementary external chemical data source 
to retrieve molecular structure representations. The file structures.py 
manages this integration through three steps:

1. **CID resolution** — each compound is first looked up in a hardcoded 
   dictionary of known PubChem compound IDs (`KNOWN_CIDS`). If absent, 
   a fallback query is sent to the PubChem API using the compound name.

2. **Image URL generation** — once the CID is retrieved, a PNG image URL 
   is constructed using the PubChem REST API, returning a 300×300 2D 
   structure representation.

3. **Equation rendering** — `render_equation()` assembles the full reaction 
   as an HTML string, placing compound images and names between `+` symbols 
   and a `→` arrow. This HTML is then passed directly to Streamlit for display.

Results from the API are cached for 24 hours using `@st.cache_data` to avoid 
redundant network calls during a session.





2. Green Chemistry Metric Calculation

The computational module (metrics.py) forms the core quantitative analysis engine of the project, automatically evaluating the environmental performance of each synthesis pathway using several green chemistry indicators:

Atom Economy is calculated as the ratio between the mass of the desired product and the total mass of reactants, providing a measure of how efficiently reactant atoms are incorporated into the final product.

$$AE = \frac{\sum M_{\text{products}}}{\sum M_{\text{reactants}}} \times 100 \quad (\%)$$

where $M$ denotes molar mass (g/mol). Note that atom economy is a theoretical metric — it does not account for reaction yield or solvent use.

The corresponding implementation is:



In [ ]:
def atom_economy(reaction):
    react_mass = total_mass(reaction.reactants)
    product_mass = total_mass(reaction.products)

    if react_mass == 0:
        return 0

    return (product_mass / react_mass) * 100



E-Factor evaluates waste generation by comparing the mass of waste produced to the actual mass of product obtained, making it a key indicator of process sustainability:

$$E\text{-}Factor = \frac{m_{\text{waste}}}{m_{\text{product}}}$$

where:

$$m_{\text{waste}} = m_{\text{reactants}} - m_{\text{product, actual}}$$

$$m_{\text{product, actual}} = m_{\text{product, theoretical}} \times \frac{Y}{100}$$

with $Y$ the reaction yield (%). A lower E-Factor indicates a cleaner, more efficient process.

Implementation:



In [ ]:
def e_factor(reaction):
    react_mass = total_mass(reaction.reactants)
    product_mass = total_mass(reaction.products)

    actual_product = product_mass * (reaction.yield_percent / 100)

    if actual_product == 0:
        return float("inf")

    waste = react_mass - actual_product
    return waste / actual_product



Solvent hazard assessment is estimated through a simplified proxy based on the average number of GHS hazard classifications associated with the solvents used in the synthesis, allowing a comparative evaluation of solvent safety.

$$SHS = \frac{\displaystyle\sum_{i=1}^{n} N_{\text{GHS},\, i}}{n}$$

where $N_{\text{GHS},\, i}$ is the number of GHS hazard pictograms associated with solvent $i$, and $n$ is the total number of solvents used in the pathway. A higher score indicates a more hazardous solvent profile.

Implementation:



In [ ]:
def solvent_toxicity(reaction):
    if not reaction.solvents:
        return 0

    total = 0

    for solvent in reaction.solvents:
        pictograms = MOLECULES[solvent.name]["ghs_pictograms"]
        total += len(pictograms)

    return total / len(reaction.solvents)




Intermediate complexity is estimated through the number of reaction intermediates, serving as a simple indicator of synthetic complexity, as more intermediates generally imply longer, less efficient, and potentially less sustainable synthesis pathways.
Similarly, synthetic complexity is estimated through the total number of synthesis steps.

Implementation:



In [ ]:
def intermediate_complexity(reaction):
    return len(reaction.intermediates)




Finally, the scoring.py file calculates a global Green Score by combining the previously described indicators into a single comparative metric. The scoring model applies weighted penalties for high waste generation, solvent hazard, synthetic complexity, and high reaction temperature as a simplified proxy for energy demand, while rewarding better atom economy. It is not a standardized scientific metric but it facilitates direct comparison between synthesis routes.

The implementation of the Green Score logic is shown below:




In [ ]:
def green_score(reaction):
    score = 75

    score -= e_factor(reaction) * 6
    score -= solvent_toxicity(reaction) * 8
    score -= reaction.steps * 5
    score -= intermediate_complexity(reaction) * 6

    if reaction.temperature > 80:
        score -= 10
    elif reaction.temperature > 50:
        score -= 5

    score += atom_economy(reaction) * 0.15

    return round(max(0, min(score, 100)), 2)




The corresponding mathematical formulation of this implementation is given below, allowing the scoring logic to be interpreted in a more formal analytical framework:

$$GS = \min\left(100,\ \max\left(0,\ S_{\text{base}} - P_{E} - P_{SHS} - P_{\text{steps}} - P_{\text{int}} - P_{T} + B_{AE}\right)\right)$$

where the penalties and bonus are defined as:

<div align="center">

| Term | Expression | Description |
|:------------------|:--------------------------------------|:---------------------------------------------|
| $S_{\text{base}}$ | $75$ | Base score |
| $P_E$ | $E\text{-}Factor \times 6$ | Penalty for waste generation |
| $P_{SHS}$ | $SHS \times 8$ | Penalty for solvent hazard |
| $P_{\text{steps}}$ | $n_{\text{steps}} \times 5$ | Penalty for synthetic complexity |
| $P_{\text{int}}$ | $n_{\text{intermediates}} \times 6$ | Penalty for intermediate complexity |
| $P_T$ | $10$ if $T>80^\circ C$, $5$ if $50<T\leq80^\circ C$ | Temperature penalty |
| $B_{AE}$ | $AE \times 0.15$ | Atom economy bonus |

</div>

$$\textit{Table 1. Mathematical definition of the Green Score weighting system.}$$

> *The Green Score is not a standardised scientific metric. Weights and thresholds were defined for the purposes of this project to facilitate comparison between synthesis routes.*

3. Green Chemistry Principles Evaluation :
 
Some principles are automatically assessed using threshold conditions derived from the previously calculated indicators. This allows the quantitative sustainability analysis to be translated into a practical interpretation in terms of green chemistry principles.

The automated evaluation relies on threshold-based decision rules implemented directly in the application logic:




In [ ]:
def compute_principles(reaction, ef, ae, hz):
    principles = []

    if ef < 10:
        principles.append("Prevention of waste")

    if ae > 50:
        principles.append("Atom economy")

    if hz < 1.5:
        principles.append("Less hazardous syntheses")

    if reaction.temperature <= 60:
        principles.append("Energy efficiency")

    return principles




However, not all green chemistry principles can be evaluated through quantitative threshold conditions alone. Some criteria rely on qualitative assumptions and are therefore predefined in `principles.py` using boolean flags assigned to each synthesis pathway. These booleans are based on simplified assumptions defined for this educational project. 

<div align="center">

| # | Principle | Method | Condition |
|:--:|:----------------------------|:--------:|:----------------------------------|
| 1 | Prevention of waste | Threshold | E-Factor < 10 |
| 2 | Atom economy | Threshold | Atom Economy > 50% |
| 3 | Less hazardous syntheses | Threshold | Hazard Score < 1.5 |
| 4 | Designing safer chemicals | Threshold | Hazard Score < 1 |
| 5 | Safer solvents | Threshold | Hazard Score < 2 |
| 6 | Energy efficiency | Threshold | Temperature ≤ 60°C |
| 7 | Renewable feedstocks | Flag | Predefined per pathway |
| 8 | Reduce derivatives | Threshold | Steps ≤ 2 and intermediates ≤ 1 |
| 9 | Catalysis | Flag | Predefined per pathway |
| 10 | Design for degradation | Flag | Predefined per pathway |
| 11 | Real-time analysis | Flag | Predefined per pathway |
| 12 | Accident prevention | Flag | Predefined per pathway |

</div>

$$\textit{Table 2. Evaluation criteria used for Green Chemistry Principle assessment.}$$

4. Graphical User Interface:

The graphical interface was implemented in Streamlit, which serves as the framework connecting the different components of the project. The app.py file loads the reaction and molecular datasets, calls the metric calculation and scoring functions, and renders the analysis results. Native Streamlit elements such as columns, selectors, and markdown containers were combined with custom HTML/CSS styling to improve the visual presentation of the dashboard. For example, custom CSS was used to design the metric cards, reaction banners, and interface layout, while embedded HTML components were used to create more refined visual elements such as the animated metric dashboards and formatted reaction displays.

The application was developed in Python 3.10 and relies on the following main dependencies:




In [ ]:
pip install streamlit
pip install pubchempy
pip install .




These dependencies support the interactive dashboard interface, project module imports, and the retrieval of molecular structure representations through the PubChem API.

## User interface

The application was designed to provide an intuitive comparison workflow for the user. Upon launching the interface, the user is first presented with the available symptom categories corresponding to the selected antihistamine use cases. After choosing a category, the relevant compounds become available for selection and comparison. Once one or several synthesis pathways are selected, the dashboard dynamically displays the corresponding analysis results. These include the overall Green Score, the calculated quantitative sustainability indicators (such as E-Factor, Atom Economy, and solvent hazard assessment), a simplified visual representation of the chemical synthesis pathway, and the qualitative evaluation of the Green Chemistry Principles. The interface was designed to provide immediate visual feedback and make the sustainability comparison accessible even to users without direct familiarity with the underlying code.

![Comparison dashboard](comparison.jpg)

*Figure 2. Comparative dashboard showing sustainability metrics, Green Score, reaction visualisation, and qualitative principle assessment in the case of diphenhydramine and doxylamine for the "Nighttime Sleep 🌙" category.*

# Results and Discussion

This section presents the computed green chemistry indicators for each 
synthesis pathway and discusses the key trends, trade-offs, and limitations 
emerging from the analysis.

### Overall Green Score Ranking

The Green Score ranking across all 17 synthesis pathways is summarised below:

<div align="center">

| Rank | Compound | Green Score | Category |
|:---:|:---------|:-----------:|:---------|
| 1 | Levocetirizine | 76.3 | Skin allergies |
| 2 | Dimenhydrinate | 74.3 | Motion sickness |
| 3 | Meclizine | 73.9 | Motion sickness |
| 4 | Hydroxyzine | 68.4 | Skin allergies |
| 5 | Desloratadine | 64.9 | Seasonal allergies |
| 6 | Promethazine | 61.3 | Nighttime sleep |
| 7 | Cyclizine | 59.8 | Motion sickness |
| 8 | Carbinoxamine | 58.0 | Cold & flu |
| 9 | Diphenhydramine | 55.0 | Nighttime sleep |
| 10 | Doxylamine | 48.9 | Nighttime sleep |
| 11 | Rupatadine | 48.5 | Skin allergies |
| 12 | Fexofenadine | 45.3 | Seasonal allergies |
| 13 | Chlorpheniramine | 41.4 | Cold & flu |
| 14 | Triprolidine | 39.8 | Cold & flu |
| 15 | Cetirizine | 34.6 | Seasonal allergies |
| 16 | Loratadine | 32.8 | Seasonal allergies |
| 17 | Brompheniramine | 28.6 | Cold & flu |

</div>

$$\textit{Table 3. Overall Green Score ranking of the 17 antihistamine synthesis pathways analysed.}$$

### Discussion

**Top performers.** 

Levocetirizine, Dimenhydrinate, and Meclizine achieve 
the three highest Green Scores (76.3, 74.3, and 73.9 respectively). These 
pathways share a favourable combination of low reaction steps (1), mild 
temperatures (25–40°C), no intermediates, and relatively low waste generation 
(E-Factor < 0.5). Levocetirizine in particular benefits from being derived 
directly from cetirizine via a single-step resolution process, which limits 
waste and complexity.

**Worst performers.** 

Brompheniramine (28.6) and Loratadine (32.8) rank at 
the bottom, driven primarily by poor reaction yields (35% and 40% respectively) 
which strongly penalise the E-Factor (3.61 and 2.95). This highlights that 
yield is one of the most influential parameters in the Green Score model.

**Trade-offs.** 

Several interesting trade-offs emerge from the data. Cyclizine 
achieves the best E-Factor among multi-step pathways (0.28) thanks to its 
exceptionally high yield (94%) and uses water as solvent — the only pathway 
to do so — resulting in a Solvent Hazard Score of 0. However, its high operating 
temperature (100°C) and the presence of an intermediate limit its overall score 
to 59.8. Fexofenadine presents the opposite case: a near-perfect atom economy 
(97%) but a low yield (55%) and 4 synthesis steps drag its score down to 45.3.

**Solvent hazard.**

 Almost all pathways use toluene, ethanol, or methanol as 
solvents, resulting in a uniform SHS of 1.0. Cetirizine is the only pathway 
using dichloromethane, a more hazardous solvent, but its SHS remains 1.0 since 
it also uses toluene (average of 1 pictogram each). This uniformity limits the 
discriminating power of the solvent indicator across this dataset.

**Limitations.** 

The Green Score is not a standardised metric and relies on 
thresholds and weights defined specifically for this project. The synthesis 
pathways were simplified, and precursor molecules were introduced to represent 
multi-step industrial routes in a single reaction object. As a result, some 
indicators — particularly atom economy — may not fully reflect the complexity 
of real industrial processes.

Additionally, the solvent hazard metric remains intentionally simplified, as it considers only the number of GHS hazard pictograms rather than weighting hazard severity or incorporating full toxicological data. Similarly, the manually assigned green chemistry principle flags introduce subjective assumptions that may not fully reflect industrial process realities.

# Conclusion

This project successfully developed an interactive dashboard for comparing 
antihistamine synthesis pathways from a green chemistry perspective. By combining 
established indicators such as E-Factor and Atom Economy with two original 
metrics — the Green Score and the 12 Principles Assessment — the tool provides 
a multi-dimensional sustainability analysis accessible to users without a 
programming background.

The results highlight that no single indicator is sufficient to characterise 
the environmental performance of a synthesis pathway, and that trade-offs 
between yield, complexity, temperature, and solvent choice must be considered 
together. Levocetirizine and Dimenhydrinate emerged as the greenest pathways 
under this framework, while Brompheniramine and Loratadine ranked lowest 
primarily due to poor reaction yields.

This work was developed in an educational context and has inherent limitations 
— simplified reaction data, non-standardised metrics, and manually assigned 
flags. Future work could address these limitations by integrating more 
comprehensive reaction datasets, validating thresholds against the green 
chemistry literature, and extending the analysis to other drug classes